> This notebook documents how I collect data for MetaTune, with a focus on sources that are legal to use and easy to justify. <br> This section also present the API I'll use for this whole project, with example on how I'll use them. This helps me learn and explore the usage of those API and also helps me validate my API keys.

Goals:
- identify reliable and legal sources
- note rate limits and license constraints
- prepare a reproducible data extraction pipeline

This notebook is not a scraping notebook. It is a data sourcing and compliance notebook.

## What is legal to use for MetaTune?

For this project, the safest sources are:
- official APIs with documented terms
- open datasets with an explicit license
- data that you generated yourself
- sources that allow reuse and storage of the fields you need

Practical rule for MetaTune:
- use the Steam Web API for metadata or allowed statistics
- use the Spotify Web API for audio features
- store only what you need
- document the source and license of each dataset

## Practices

Good practice for this project:
- prefer official APIs with clear terms of use
- use open datasets with an explicit license
- use databases that are explicitly shared for reuse
- avoid sources that do not clearly allow extraction or reuse

For MetaTune, the useful sources are:
- Steam Web API for the most played games
- Spotify Web API via Spotipy for albums and audio features
- public datasets with licenses on Kaggle or GitHub
- manually curated game lists

What to avoid:
- massive website scraping
- redistribution of restricted data
- datasets with unclear or missing licensing

In this part we will:

1. list sources with their licensing
2. store useful columns
3. load a first sample of data to see what we're dealing with
4. normalize the data when necessary
5. export a CSV or data file for the engine and future notebooks

#### **Steam API example workflow:**
https://steamcommunity.com/dev

Before we have a Steam key, this notebook uses a manual seed list of Life is Strange games.

Once the key is available, the next step will be:
- send each game title to the Steam search or app-details endpoint
- compare the returned app ID with the manual seed list
- store the verified metadata in a clean dataframe
- keep only fields that are allowed by the API terms

With this API we can :
- Compare the returned `appID` with your manual seed list; mark matches as verified.  
- Collect and store verified metadata in a clean dataframe, keeping only allowed fields (e.g., title, appid, release_date, genres/tags, developer, publisher, store_url).  
- Persist the dataframe (e.g., `data/steam_metadata.csv`) with a retrieval timestamp and source column.  
- Respect rate limits: implement exponential backoff, small delays, and local caching to avoid re-querying.  
- Keep `STEAM_API_KEY` private (.env, excluded by .gitignore) and do not commit keys or user-identifying data.

#### **Spotipy license:**
https://spotipy.readthedocs.io/en/2.26.0/

Spotipy is distributed under the MIT License.

In practice, that means:
- we can use, modify, and redistribute it
- we should keep the copyright notice and license text in substantial copies
- it comes with no warranty

For MetaTune, Spotipy is a safe choice for Python access to the Spotify Web API, as long as we also respect Spotify's own API terms and usage limits.

#### **Example of Spotipy Usage: Life is Strange**

> For the rest of the notebook we're gonna pratice the usage of those API to familiarize with those.
> We're gonna be using a dataframe made by myself to find albums of one of my favorite game license : Life Is Strange !

In [ ]:
!pip install pandas spotipy python-steam-api

In [ ]:
import pandas as pd

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 120)

life_is_strange_games = pd.DataFrame([
    {
        'game_title': 'Life is Strange',
        'steam_app_id': None,
        'source': 'manual example list',
        'status': 'example only',
    },
    {
        'game_title': 'Life is Strange: Before the Storm',
        'steam_app_id': None,
        'source': 'manual example list',
        'status': 'example only',
    },
    {
        'game_title': 'Life is Strange 2',
        'steam_app_id': None,
        'source': 'manual example list',
        'status': 'example only',
    },
    {
        'game_title': 'Life is Strange: True Colors',
        'steam_app_id': None,
        'source': 'manual example list',
        'status': 'example only',
    },
    {
        'game_title': 'Life is Strange: Double Exposure',
        'steam_app_id': None,
        'source': 'manual example list',
        'status': 'example only',
    },
    {
        'game_title': 'Life is Strange: Reunion',
        'steam_app_id': None,
        'source': 'manual example list',
        'status': 'example only',
    },
])

life_is_strange_games

In [ ]:
# Let's see what sp.search actually returns
# We'll do a simple search and explore the structure

test_search = sp.search(q='Life is Strange soundtrack', type='album', limit=1)

print("///**** SPOTIFY SEARCH RESPONSE STRUCTURE ****///\n")
print("Top-level keys:")
print(test_search.keys())
print("\n")

print("Albums section keys:")
print(test_search['albums'].keys())
print("\n")

print("First album (raw):")
first_album_raw = test_search['albums']['items'][0]
print(f"  - name: {first_album_raw['name']}")
print(f"  - id: {first_album_raw['id']}")
print(f"  - artists: {[artist['name'] for artist in first_album_raw['artists']]}")
print(f"  - release_date: {first_album_raw['release_date']}")
print(f"  - total_tracks: {first_album_raw['total_tracks']}")
print(f"  - images: {len(first_album_raw['images'])} available")
print("\n")

print("Full album object keys:")
print(first_album_raw.keys())
print("\n")

print("What we extracted for our example:")
print({
    'game_title': 'Life is Strange',
    'album_name': first_album_raw['name'],
    'album_id': first_album_raw['id'],
    'artists': ', '.join([artist['name'] for artist in first_album_raw['artists']]),
})

In [ ]:
import spotipy
import os
import pandas as pd
from spotipy.oauth2 import SpotifyClientCredentials
from dotenv import load_dotenv

load_dotenv()

# We set up the credentials needed, those are on the .env file.
CLIENT_ID = os.getenv("SPOTIFY_CLIENT_ID")
CLIENT_SECRET = os.getenv("SPOTIFY_CLIENT_SECRET")

# We initiate our auth manager to start queries
auth_manager = SpotifyClientCredentials(client_id=CLIENT_ID, client_secret=CLIENT_SECRET)
sp = spotipy.Spotify(auth_manager=auth_manager, requests_timeout=20)

# Let's keep up with the previous example ! 
# We're gonna look for Life is Strange soundtracks using the dataset we created
lis_titles = life_is_strange_games["game_title"]

# We're gonna collect all albums from all searches with their associated game
all_albums_with_game = []
print("Searching for Life is Strange soundtracks...\n")

for title in lis_titles:
    # We keep the search limit low (2) to minimize the data payload
    search_results = sp.search(q=title+' soundtrack', type='album', limit=2)
    albums = search_results['albums']['items']
    
    # Let's store each album with the game it came from
    for album in albums:
        all_albums_with_game.append({
            'game_title': title,
            'album_name': album['name'],
            'album_id': album['id'],
            'artists': ', '.join([artist['name'] for artist in album['artists']]),
        })

print(f"Total albums found: {len(all_albums_with_game)}\n")

albums_df = pd.DataFrame(all_albums_with_game)
albums_df_unique = albums_df.drop_duplicates(subset=['album_id'])

print(f"Unique albums: {len(albums_df_unique)}\n")

# Now we're gonna fetch track metadata AND audio features from the first album's tracks
# We use sp.audio_features() to get danceability, energy, valence, acousticness, etc.
if len(albums_df_unique) > 0:
    first_album = albums_df_unique.iloc[0]
    
    # We fetch the first 10 tracks
    tracks = sp.album_tracks(first_album['album_id'], limit=10)
    
    track_data_list = []
    
    # Get all track IDs first
    track_ids = [track['id'] for track in tracks['items']]
    
    # Fetch audio features for all tracks in one batch call (more efficient)
    audio_features = sp.audio_features(track_ids)
    
    for i, track in enumerate(tracks['items']):
        # Get the audio features for this track
        features = audio_features[i]
        
        if features:  # Only add if we got valid audio features
            track_data_list.append({
                'game_title': first_album['game_title'],
                'album_name': first_album['album_name'],
                'track_name': track['name'],
                'duration_ms': track['duration_ms'],
                'track_number': track['track_number'],
                'explicit': track['explicit'],
                # AUDIO FEATURES - The vibe metrics!
                'danceability': features['danceability'],
                'energy': features['energy'],
                'valence': features['valence'],  # Musical positiveness
                'acousticness': features['acousticness'],
                'instrumentalness': features['instrumentalness'],
                'liveness': features['liveness'],
                'speechiness': features['speechiness'],
                'tempo': features['tempo'],
            })
    
    audio_df = pd.DataFrame(track_data_list)
    print(f"Successfully extracted {len(audio_df)} tracks with audio features:\n")
    print(audio_df[['track_name', 'danceability', 'energy', 'valence', 'acousticness']])

#### From Spotipy to Audio Features via RapidAPI
(https://rapidapi.com/soundnet-soundnet-default/api/track-analysis)

Spotify deprecated the `audio-features` endpoint and now requires higher access tier authentication. The endpoint returns 403 Forbidden for standard OAuth tokens.

Instead, we use the RapidAPI Track Analysis API by SoundNet. This API provides the same audio features we need:
- energy, danceability, acousticness, liveness, speechiness
- additional useful metrics: key, mode, tempo, popularity, loudness
- direct support for Spotify track IDs

This approach:
- avoids authentication issues with deprecated Spotify endpoints
- provides more features than the old endpoint
- remains within free tier rate limits for development
- is reproducible and well-documented

The audio features are the foundation of MetaTune's similarity calculations. We aggregate them by game to create mood profiles, then use Euclidean distance to rank games by musical similarity.

## Legal Status: RapidAPI Track Analysis API

### SoundNet API Terms

SoundNet's Track Analysis API is offered through RapidAPI's marketplace. Key points:

- The API is explicitly designed for educational and development use
- Free tier includes requests for development (500/month on BASIC plan)
- Commercial use requires the PRO plan ($9.99/month or higher)
- API returns computed audio analysis (features), not copyrighted audio content
- Redistribution of analysis results follows SoundNet's license terms

### Spotify Track ID Usage

Using Spotify track IDs with this API is legitimate because:
- We only retrieve metadata (audio features), not audio files from Spotify
- Audio features are analytical/computational results, not protected content
- This mirrors how Last.fm, Genius, and other music services provide supplementary analysis
- We are not redistributing Spotify's copyrighted catalog

### MetaTune Use Case - Legal Status

MetaTune's use of this data is compliant for:
- Similarity recommendation based on mood analysis
- Research and development on game soundtrack characteristics
- Educational project for game recommendation systems
- Personal non-commercial use

Must follow:
- Do not redistribute raw Spotify metadata or bulk feature tables directly
- Do not claim SoundNet analysis as original work
- Respect API rate limits and subscription tier
- Maintain attribution: "Audio analysis via SoundNet"

### Compliance Summary

This approach meets requirements of:
- Spotify's API terms (approved APIs only, no scraping)
- SoundNet/RapidAPI terms (permitted use within subscription tier)
- Fair use principles (transformative use for recommendation analysis)

In [ ]:
import requests
import os
import pandas as pd
from dotenv import load_dotenv
import time
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

load_dotenv()

# RapidAPI credentials for Track Analysis API by SoundNet
RAPIDAPI_KEY = os.getenv("RAPIDAPI_KEY")
RAPIDAPI_HOST = "track-analysis.p.rapidapi.com"

if not RAPIDAPI_KEY:
    print("WARNING: RAPIDAPI_KEY not found in .env file")
    print("Get one free at: https://rapidapi.com/SoundNet/api/track-analysis")
    print("Add this to your .env file: RAPIDAPI_KEY=your_key_here\n")
else:
    print("RapidAPI key loaded\n")

retry_strategy = Retry(
    total=5,
    backoff_factor=2,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"],
    respect_retry_after_header=True,
)
session = requests.Session()
session.mount("https://", HTTPAdapter(max_retries=retry_strategy))

def get_audio_features_rapidapi(spotify_track_id):
    url = f"https://{RAPIDAPI_HOST}/pktx/spotify/{spotify_track_id}"

    headers = {
        "x-rapidapi-key": RAPIDAPI_KEY,
        "x-rapidapi-host": RAPIDAPI_HOST
    }

    try:
        response = session.get(url, headers=headers, timeout=30)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.RequestException as e:
        print(f"Error fetching features for {spotify_track_id}: {e}")
        return None

# Fetch a small sample first so we do not hit the free-tier rate limit too hard
if len(albums_df_unique) > 0 and RAPIDAPI_KEY:
    first_album = albums_df_unique.iloc[0]
    tracks = sp.album_tracks(first_album['album_id'], limit=10)
    sample_tracks = tracks['items'][:3]

    track_data_list = []
    print(f"Fetching audio features for {len(sample_tracks)} sample tracks from RapidAPI\n")

    for i, track in enumerate(sample_tracks):
        features = get_audio_features_rapidapi(track['id'])

        if features and 'energy' in features:
            track_data_list.append({
                'game_title': first_album['game_title'],
                'album_name': first_album['album_name'],
                'track_name': track['name'],
                'duration_ms': track['duration_ms'],
                'track_number': track['track_number'],
                'explicit': track['explicit'],
                'key': features.get('key'),
                'mode': features.get('mode'),
                'camelot': features.get('camelot'),
                'tempo': features.get('tempo'),
                'popularity': features.get('popularity'),
                'energy': features.get('energy'),
                'danceability': features.get('danceability'),
                'happiness': features.get('happiness'),
                'acousticness': features.get('acousticness'),
                'instrumentalness': features.get('instrumentalness'),
                'liveness': features.get('liveness'),
                'speechiness': features.get('speechiness'),
                'loudness': features.get('loudness'),
            })

        if i < len(sample_tracks) - 1:
            time.sleep(2)

    audio_df_rapidapi = pd.DataFrame(track_data_list)
    print(f"Extracted {len(audio_df_rapidapi)} tracks with audio features\n")

    if not audio_df_rapidapi.empty:
        print(audio_df_rapidapi[['track_name', 'danceability', 'energy', 'happiness', 'acousticness', 'tempo']])
    else:
        print("No audio features were returned. The API is rate limited or the free tier is exhausted.")
else:
    print("Skipping RapidAPI call - check if album data and API key are available")

> Example of usage of Steam API :

In [ ]:
import os
from dotenv import load_dotenv
from steam_web_api import Steam

load_dotenv()

STEAM_KEY = os.environ.get("STEAM_API_KEY")
steam = Steam(STEAM_KEY)

MY_STEAM_ID = os.getenv("MY_STEAM_ID")
if not MY_STEAM_ID:
    MY_STEAM_ID = input("Enter your Steam ID, or press Enter to use a redacted example: ").strip()

if not MY_STEAM_ID:
    print("No Steam ID provided, so this cell uses a redacted example only.")
else:
    user = steam.users.get_user_details(MY_STEAM_ID)

    owned_games = steam.users.get_owned_games(MY_STEAM_ID)
    print("Owned games: ", owned_games["game_count"])

    print(owned_games["games"])
    # This is gonna be very useful to fetch the user's most played games and then do a
    # centroid based on those games' albums
    print(f"\n 5 games for user {MY_STEAM_ID} :")
    for i in range(min(5, len(owned_games["games"]))):
        print(owned_games["games"][i]["name"])
